# LAB 08 — PHÂN CỤM K-MEANS

**Sinh viên:** Lê Quyết Tiến  
**MSSV:** 2474802010386  
**Môn:** Học Máy Và Ứng Dụng  
**LHP:** 261_71ITAI41203_0101  
**GVHD:** ThS. Nguyễn Thái Anh

Notebook này thực hiện đầy đủ phần hướng dẫn trước mục **Bài tập** của Lab 08.


In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import load_sample_image, load_digits
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"
FIGURES = ROOT / "figures"
OUTPUTS = ROOT / "outputs"
FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")
print("Thu muc Lab 08:", ROOT.resolve())


Thu muc Lab 08: C:\Users\lequy\Documents\Codex\2026-09-22\o\work\lab08_build\Lab08_LeQuyetTien


## 1. Tự viết thuật toán K-Means bằng NumPy


In [2]:
P = np.array([[1, 1], [1.5, 2], [3, 4], [5, 7],
              [3.5, 5], [4.5, 5], [3.5, 4.5]], dtype=float)

def kmeans_tu_viet(X, k=2, max_iter=100, tol=1e-8):
    tam = X[[0, 1]].copy()
    for vong in range(max_iter):
        khoang_cach = np.linalg.norm(X[:, None, :] - tam[None, :, :], axis=2)
        nhan = np.argmin(khoang_cach, axis=1)
        tam_moi = np.array([X[nhan == j].mean(axis=0) for j in range(k)])
        if np.linalg.norm(tam_moi - tam) < tol:
            tam = tam_moi
            break
        tam = tam_moi
    inertia = np.sum((X - tam[nhan]) ** 2)
    return tam, nhan, inertia, vong + 1

tam_np, nhan_np, inertia_np, so_vong = kmeans_tu_viet(P)
mo_hinh = KMeans(n_clusters=2, init=P[[0, 1]], n_init=1, random_state=0).fit(P)

print("Tam tu viet:\n", tam_np)
print("Nhan:", nhan_np)
print("So vong lap:", so_vong)
print("Inertia tu viet:", round(inertia_np, 3))
print("Inertia sklearn:", round(mo_hinh.inertia_, 3))

plt.figure(figsize=(6, 4))
plt.scatter(P[:, 0], P[:, 1], c=nhan_np, cmap="viridis", s=80)
plt.scatter(tam_np[:, 0], tam_np[:, 1], c="red", marker="X", s=220, label="Tam cum")
plt.title("K-Means tu viet")
plt.legend(); plt.tight_layout()
plt.savefig(FIGURES / "k1_tu_viet.png", dpi=150)
plt.show()


Tam tu viet:
 [[1.25 1.5 ]
 [3.9  5.1 ]]
Nhan: [0 0 1 1 1 1 1]
So vong lap: 3
Inertia tu viet: 8.525
Inertia sklearn: 8.525


C:\Users\lequy\AppData\Local\Temp\ipykernel_34560\2613663024.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Nhận xét:** Thuật toán tự viết hội tụ về hai tâm xấp xỉ `(1.25, 1.50)` và `(3.90, 5.10)`. Inertia trùng với scikit-learn (khoảng `8.525`), cho thấy các bước gán cụm và cập nhật tâm được cài đặt đúng.


## 2. Phân khúc khách hàng


In [3]:
khach_hang = pd.read_csv(DATA / "khach_hang.csv")
X_kh = khach_hang[["thu_nhap", "diem_chi_tieu"]]
bo_chuan_hoa = StandardScaler()
X_kh_chuan = bo_chuan_hoa.fit_transform(X_kh)

kmeans5 = KMeans(n_clusters=5, n_init=10, random_state=0)
khach_hang["cum"] = kmeans5.fit_predict(X_kh_chuan)
tam_goc = bo_chuan_hoa.inverse_transform(kmeans5.cluster_centers_)

bang_tam = pd.DataFrame(tam_goc, columns=["thu_nhap", "diem_chi_tieu"])
bang_tam["so_khach"] = khach_hang["cum"].value_counts().sort_index().values
bang_tam["tuoi_trung_binh"] = khach_hang.groupby("cum")["tuoi"].mean().values
print("Inertia:", round(kmeans5.inertia_, 3))
print(bang_tam.round(2))

khach_moi = pd.DataFrame([[29, 85], [9, 75]], columns=X_kh.columns)
cum_moi = kmeans5.predict(bo_chuan_hoa.transform(khach_moi))
print("\nKhach hang moi va cum du doan:")
print(khach_moi.assign(cum=cum_moi))

plt.figure(figsize=(7, 5))
plt.scatter(khach_hang.thu_nhap, khach_hang.diem_chi_tieu,
            c=khach_hang.cum, cmap="tab10", alpha=.75)
plt.scatter(tam_goc[:, 0], tam_goc[:, 1], marker="X", s=250, c="black", label="Tam cum")
plt.xlabel("Thu nhap"); plt.ylabel("Diem chi tieu")
plt.title("Phan khuc khach hang voi K=5")
plt.legend(); plt.tight_layout()
plt.savefig(FIGURES / "k2_khach_hang.png", dpi=150)
plt.show()


Inertia:

C:\Users\lequy\AppData\Local\Temp\ipykernel_34560\269645796.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 23.017
   thu_nhap  diem_chi_tieu  so_khach  tuoi_trung_binh
0      7.48          80.20        35            24.94
1     30.13          19.97        35            42.86
2     17.39          49.30        60            40.45
3      8.04          18.85        40            44.88
4     30.21          81.53        40            32.62

Khach hang moi va cum du doan:
   thu_nhap  diem_chi_tieu  cum
0        29             85    4
1         9             75    0


**Nhận xét:** Năm cụm biểu diễn các nhóm khách hàng có mức thu nhập và chi tiêu khác nhau. Hai khách hàng mới được gán vào nhóm gần tâm nhất sau khi dùng đúng bộ chuẩn hóa đã huấn luyện.


## 3. Chọn số cụm K bằng Elbow và Silhouette


In [4]:
ket_qua_k = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_kh_chuan)
    sil = np.nan if k == 1 else silhouette_score(X_kh_chuan, km.labels_)
    ket_qua_k.append((k, km.inertia_, sil))

chon_k = pd.DataFrame(ket_qua_k, columns=["K", "inertia", "silhouette"])
print(chon_k.round(4).to_string(index=False))
best_k = int(chon_k.loc[chon_k.silhouette.idxmax(), "K"])
print("\nK tot nhat theo silhouette:", best_k)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(chon_k.K, chon_k.inertia, "o-")
ax[0].set(title="Elbow", xlabel="K", ylabel="Inertia")
ax[1].plot(chon_k.K[1:], chon_k.silhouette[1:], "o-", color="darkorange")
ax[1].set(title="Silhouette", xlabel="K", ylabel="He so")
plt.tight_layout(); plt.savefig(FIGURES / "k3_chon_k.png", dpi=150); plt.show()


 K  inertia  silhouette
 1 420.0000         NaN
 2 250.3985      0.4300
 3 152.5400      0.5217
 4  73.7574      0.6256
 5  23.0167      0.7227
 6  20.8739      0.6508
 7  19.0119      0.5413
 8  17.4235      0.4538
 9  15.8928      0.4713
10  14.4467      0.4254

K tot nhat theo silhouette: 5


C:\Users\lequy\AppData\Local\Temp\ipykernel_34560\1235680739.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIGURES / "k3_chon_k.png", dpi=150); plt.show()


**Nhận xét:** Đường Elbow giảm mạnh trước K=5 rồi chậm lại; đồng thời Silhouette đạt cực đại tại K=5. Vì vậy K=5 là lựa chọn hợp lý cho dữ liệu khách hàng.


## 4. Ba cái bẫy thường gặp của K-Means


In [5]:
# Bay 1: khong chuan hoa
km_raw = KMeans(n_clusters=5, n_init=10, random_state=0).fit(X_kh)
sil_raw = silhouette_score(X_kh, km_raw.labels_)
sil_scaled = silhouette_score(X_kh_chuan, kmeans5.labels_)

# Bay 2: khoi tao kem va chi chay mot lan
inertia_random = []
for seed in range(20):
    inertia_random.append(KMeans(n_clusters=5, init="random", n_init=1,
                                 random_state=seed).fit(X_kh_chuan).inertia_)

# Bay 3: cum khong tron (hai vang trang)
moon = pd.read_csv(DATA / "hai_vang_trang.csv")
X_moon = moon[["x1", "x2"]].to_numpy()
y_moon = moon["nhan"].to_numpy()
nhan_km = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X_moon)

print(f"Silhouette khong chuan hoa: {sil_raw:.3f}")
print(f"Silhouette co chuan hoa:   {sil_scaled:.3f}")
print("Inertia voi khoi tao random (min/max):",
      round(min(inertia_random), 3), round(max(inertia_random), 3))
print("ARI cua K-Means tren hai vang trang:", round(adjusted_rand_score(y_moon, nhan_km), 3))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(X_moon[:, 0], X_moon[:, 1], c=y_moon, cmap="coolwarm", s=20)
ax[0].set_title("Nhan dung")
ax[1].scatter(X_moon[:, 0], X_moon[:, 1], c=nhan_km, cmap="coolwarm", s=20)
ax[1].set_title("K-Means")
plt.tight_layout(); plt.savefig(FIGURES / "k4_ba_cai_bay.png", dpi=150); plt.show()


Silhouette khong chuan hoa: 0.666
Silhouette co chuan hoa:   0.723
Inertia voi khoi tao random (min/max): 23.017 118.162
ARI cua K-Means tren hai vang trang: 0.247


C:\Users\lequy\AppData\Local\Temp\ipykernel_34560\2940478915.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIGURES / "k4_ba_cai_bay.png", dpi=150); plt.show()


**Nhận xét:** Cần chuẩn hóa khi các thuộc tính khác thang đo; nên dùng `k-means++` và nhiều lần khởi tạo; K-Means không phù hợp với cụm cong như hai vành trăng vì thuật toán ưu tiên các cụm gần hình cầu.


## 5. Nén ảnh bằng K-Means


In [6]:
anh = load_sample_image("china.jpg")
pixel = anh.reshape(-1, 3).astype(float)
rng = np.random.default_rng(0)
mau_huan_luyen = pixel[rng.choice(len(pixel), size=5000, replace=False)]

print("Kich thuoc anh:", anh.shape)
print("So pixel:", len(pixel))
print("So mau khac nhau:", len(np.unique(pixel, axis=0)))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, k in zip(axes, [4, 16, 64]):
    km_anh = KMeans(n_clusters=k, n_init=3, random_state=0).fit(mau_huan_luyen)
    nhan_pixel = km_anh.predict(pixel)
    pixel_nen = km_anh.cluster_centers_[nhan_pixel]
    sai_so = np.mean(np.linalg.norm(pixel - pixel_nen, axis=1))
    anh_nen = np.clip(pixel_nen.reshape(anh.shape), 0, 255).astype(np.uint8)
    ax.imshow(anh_nen); ax.axis("off"); ax.set_title(f"K={k}, loi={sai_so:.2f}")

plt.tight_layout(); plt.savefig(FIGURES / "k5_nen_anh.png", dpi=150); plt.show()


Kich thuoc anh: (427, 640, 3)
So pixel: 273280
So mau khac nhau: 96615


C:\Users\lequy\AppData\Local\Temp\ipykernel_34560\3178653645.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIGURES / "k5_nen_anh.png", dpi=150); plt.show()


**Nhận xét:** K càng lớn thì ảnh càng giữ được nhiều màu và sai số tái tạo càng nhỏ, đổi lại cần lưu nhiều tâm màu hơn và thời gian tính tăng.
